In [8]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from cellpose import io

# -------------------------------
# Configuration and Setup
# -------------------------------
base_input_folder = 'TIFs'  # Base folder containing subfolders with images and segmentation files
output_folder_images = 'trashi'       # For saving overlay images
output_folder_outlines = 'TIFs'    # For saving cell outlines
output_folder_filtered_npy = 'TIFs'  # For saving filtered segmentation files

# Create output directories if they don't exist
os.makedirs(output_folder_images, exist_ok=True)
os.makedirs(output_folder_outlines, exist_ok=True)
os.makedirs(output_folder_filtered_npy, exist_ok=True)

# Recursively find all segmentation files (assumed to end with '_seg.npy')
segmentation_files = glob.glob(os.path.join(base_input_folder, '**', '*_seg.npy'), recursive=True)

# Define HSV thresholds and green pixel ratio threshold
lower_green = np.array([0, 200, 45])
upper_green = np.array([12, 255, 255])
green_pixel_ratio_threshold = 0.50  # 50% or more pixels must be green

# -------------------------------
# Process each file pair in the folder tree
# -------------------------------
for seg_file in segmentation_files:
    # Determine the base name (e.g., "image_seg.npy" -> "image")
    base_name = os.path.basename(seg_file).replace('_seg.npy', '')
    
    # The corresponding image is assumed to be in the same folder with a .tif extension
    img_file = os.path.join(os.path.dirname(seg_file), base_name + '.tif')
    
    # Check if the corresponding image file exists
    if not os.path.exists(img_file):
        print(f"Image file {img_file} not found for segmentation file {seg_file}. Skipping.")
        continue

    # Determine subfolder relative to the base_input_folder for output organization
    subfolder = os.path.relpath(os.path.dirname(seg_file), base_input_folder)
    out_img_folder = os.path.join(output_folder_images, subfolder)
    out_outline_folder = os.path.join(output_folder_outlines, subfolder)
    out_filtered_npy_folder = os.path.join(output_folder_filtered_npy, subfolder)
    os.makedirs(out_img_folder, exist_ok=True)
    os.makedirs(out_outline_folder, exist_ok=True)
    os.makedirs(out_filtered_npy_folder, exist_ok=True)
    
    print(f"Processing {base_name} in folder {subfolder}...")

    # -------------------------------
    # Part 1: Load data and build the mask dictionary
    # -------------------------------
    dat = np.load(seg_file, allow_pickle=True).item()
    img = io.imread(img_file)

    # Ensure the image is in RGB (if grayscale, convert it)
    if img.ndim == 2 or (img.ndim == 3 and img.shape[2] == 1):
        img = np.stack([img.squeeze()] * 3, axis=-1)

    # -------------------------------
    # Save original image as PNG
    # -------------------------------
    original_png_filename = base_name + '.png'
    original_png_filepath = os.path.join(out_outline_folder, original_png_filename)
    plt.imsave(original_png_filepath, img)
    print(f" - Original image saved as PNG to: {original_png_filepath}")

    masks = dat['masks']

    # Create a dictionary: key -> tuple of (y,x) coordinates for the mask pixels,
    # value -> average RGB value (for visualization if desired)
    mask_dict = {}
    labels = np.unique(masks)
    for label in labels:
        if label == 0:  # Skip background
            continue
        cell_mask = masks == label
        coords = np.argwhere(cell_mask)
        coords_tuple = tuple(map(tuple, coords))
        avg_rgb = np.mean(img[cell_mask], axis=0)
        mask_dict[coords_tuple] = avg_rgb

    # -------------------------------
    # Part 2: Filter cells using HSV thresholds
    # -------------------------------
    # Convert the image from RGB to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    # Create a binary mask for green regions
    hsv_green_mask = cv2.inRange(hsv, lower_green, upper_green)

    filtered_dict = {}
    # Track which labels (cell IDs) to keep
    green_labels = set()
    
    for coords, avg_rgb in mask_dict.items():
        coords_array = np.array(coords)  # shape (N, 2)
        # Extract the green mask values for the cell's pixels
        green_vals = hsv_green_mask[coords_array[:, 0], coords_array[:, 1]]
        ratio_green = np.sum(green_vals == 255) / len(green_vals)
        
        # Find which label this cell corresponds to
        y, x = coords_array[0]  # Use the first coordinate to identify the label
        label = masks[y, x]
        
        if ratio_green >= green_pixel_ratio_threshold:
            filtered_dict[coords] = avg_rgb
            green_labels.add(label)

    # -------------------------------
    # NEW Part: Modify the .npy file to only keep green cell masks
    # -------------------------------
    # Create a copy of the original masks
    filtered_masks = masks.copy()
    
    # Set all non-green cell labels to zero in the mask
    for label in labels:
        if label != 0 and label not in green_labels:  # Skip background and green cells
            filtered_masks[filtered_masks == label] = 0
    
    # Create a copy of the original segmentation data
    filtered_dat = dat.copy()
    # Update the masks in the segmentation data
    filtered_dat['masks'] = filtered_masks
    
    # Save the modified segmentation data
    filtered_npy_file = os.path.join(out_filtered_npy_folder, base_name + '_seg.npy')
    np.save(filtered_npy_file, filtered_dat)
    
    print(f" - Filtered segmentation saved to: {filtered_npy_file}")
    print(f" - Kept {len(green_labels)} green cells out of {len(labels) - 1} total cells")

    # -------------------------------
    # Part 3: Create a black background with just the green cells and save the image
    # -------------------------------
    # Create a black background image (all zeros)
    green_cells_image = np.zeros(img.shape, dtype=np.uint8)
    
    # Set the green cells to green color (0, 255, 0) on the black background
    for coords in filtered_dict.keys():
        coords_array = np.array(coords)
        # Use a pure green color for all cells
        green_cells_image[coords_array[:, 0], coords_array[:, 1]] = [255, 0, 0]  # RGB for green

    # Save the green cells image to the output folder (preserving subfolder structure)
    output_green_cells_file = os.path.join(out_img_folder, base_name + '_green_cells.png')
    
    # Setup the figure with tight layout and no edge padding
    plt.figure(figsize=(10, 10))
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)  # Remove internal padding
    plt.imshow(green_cells_image)
    plt.axis('off')  # Turn off axis
    
    # Save with tight bounding box, no padding, and black background
    plt.savefig(output_green_cells_file, 
                bbox_inches='tight',  # Remove any extra whitespace
                pad_inches=0,         # Remove all padding
                facecolor='black',    # Make figure background black
                dpi=300)              # Higher DPI for better quality
    plt.close()

    # -------------------------------
    # Part 4: Extract cell outlines and write to file
    # -------------------------------
    outline_file = os.path.join(out_outline_folder, base_name + '_outlines.txt')
    with open(outline_file, "w") as f:
        for coords in filtered_dict.keys():
            # Create a binary image for this cell mask
            cell_mask_img = np.zeros(masks.shape, dtype=np.uint8)
            coords_np = np.array(coords)
            cell_mask_img[coords_np[:, 0], coords_np[:, 1]] = 255

            # Find contours; using RETR_EXTERNAL to get only the outer boundary
            contours, _ = cv2.findContours(cell_mask_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) == 0:
                continue

            # Choose the largest contour (in case there are multiple)
            largest_contour = max(contours, key=cv2.contourArea)
            largest_contour = largest_contour.squeeze()
            # Ensure the contour is 2D (in case it's only a single point)
            if largest_contour.ndim == 1:
                largest_contour = np.expand_dims(largest_contour, axis=0)

            # Format the contour as "x,y,x,y,..." (note: OpenCV contours are in (x,y) order)
            coord_pairs = [f"{pt[0]},{pt[1]}" for pt in largest_contour]
            outline_str = ",".join(coord_pairs)
            f.write(outline_str + "\n")

    print(f"Finished processing {base_name} in folder {subfolder}.")
    print(f" - Green cells image saved to: {output_green_cells_file}")
    print(f" - Outlines saved to: {outline_file}")

Processing 13 in folder 9TIF...
 - Original image saved as PNG to: TIFs/9TIF/13.png
 - Filtered segmentation saved to: TIFs/9TIF/13_seg.npy
 - Kept 1 green cells out of 1 total cells
Finished processing 13 in folder 9TIF.
 - Green cells image saved to: trashi/9TIF/13_green_cells.png
 - Outlines saved to: TIFs/9TIF/13_outlines.txt
Processing 16 in folder 9TIF...
 - Original image saved as PNG to: TIFs/9TIF/16.png
 - Filtered segmentation saved to: TIFs/9TIF/16_seg.npy
 - Kept 0 green cells out of 0 total cells
Finished processing 16 in folder 9TIF.
 - Green cells image saved to: trashi/9TIF/16_green_cells.png
 - Outlines saved to: TIFs/9TIF/16_outlines.txt
Processing 9 in folder 9TIF...
 - Original image saved as PNG to: TIFs/9TIF/9.png
 - Filtered segmentation saved to: TIFs/9TIF/9_seg.npy
 - Kept 6 green cells out of 6 total cells
Finished processing 9 in folder 9TIF.
 - Green cells image saved to: trashi/9TIF/9_green_cells.png
 - Outlines saved to: TIFs/9TIF/9_outlines.txt
Processing

In [ ]:
data = np.load("A1/t1/4_filtered_seg.npy", allow_pickle=True)
data

array({'img': array([[[0, 5, 2],
        [0, 0, 0],
        [5, 0, 0],
        ...,
        [0, 0, 0],
        [2, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       ...,

       [[0, 0, 0],
        [0, 4, 1],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[3, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]], dtype=uint8), 'outlines': array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
 